
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning" style="width: 600px">
</div>

<i18n value="8c6d3ef3-e44b-4292-a0d3-1aaba0198525"/>

# ????????? (Data Cleansing)

???Spark?????<a href="http://insideairbnb.com/get-the-data.html" target="_blank">Inside Airbnb</a>?SF Airbnb??????????????????????????????

<img src="https://files.training.databricks.com/images/301/sf.jpg" style="height: 200px; margin: 10px; border: 1px solid #ddd; padding: 10px"/>

## ![Spark Logo Tiny](https://files.training.databricks.com/images/105/logo_spark_tiny.png)?????????????: <br>
 - ??????
 - ?????????

In [ ]:
%run "./Includes/Classroom-Setup"

<i18n value="969507ea-bffc-4255-9a99-2306a594625f"/>

Airbnb???????????????????

In [ ]:
file_path = f"{DA.paths.datasets}/airbnb/sf-listings/sf-listings-2019-03-06.csv"

raw_df = spark.read.csv(file_path, header="true", inferSchema="true", multiLine="true", escape='"')

display(raw_df)

In [ ]:
raw_df.columns

<i18n value="94856418-c319-4915-a73e-5728fcd44101"/>

?????????????????????????????????????????????

In [ ]:
columns_to_keep = [
    "host_is_superhost",
    "cancellation_policy",
    "instant_bookable",
    "host_total_listings_count",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "bed_type",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "price"
]

base_df = raw_df.select(columns_to_keep)
base_df.cache().count()
display(base_df)

<i18n value="a12c5a59-ad1c-4542-8695-d822ec10c4ca"/>

### ??????? (Fixing Data Types)

?????????????? **`????(price)`** ??????????????????????????????????? ??(double ?)???????????????? 

?????????????

In [ ]:
from pyspark.sql.functions import col, translate

fixed_price_df = base_df.withColumn("price", translate(col("price"), "$,", "").cast("double"))

display(fixed_price_df)

<i18n value="4ad08138-4563-4a93-b038-801832c9bc73"/>

### ???? (Summary statistics)

?????????2?????????????
* **`describe`** : count, mean, stddev, min, max 
* **`summary`** : ???describe??? + ?????(IQR : interquartile range)

**??** : ?????? IQR/????????????????

In [ ]:
display(fixed_price_df.describe())

In [ ]:
display(fixed_price_df.summary())

<i18n value="bd55efda-86d0-4584-a6fc-ef4f221b2872"/>

### Dbutils ???????? (Dnutils Data Summary)

 **`dbutils.data.summarize`** ??????????????????????????????????

In [ ]:
dbutils.data.summarize(fixed_price_df)

<i18n value="e9860f92-2fbe-4d23-b728-678a7bb4734e"/>

### ??????? (Geeting rid of exterme values)

 **`price`** ????? *???* ? *???* ?????????

In [ ]:
display(fixed_price_df.select("price").describe())

<i18n value="4a8fe21b-1dac-4edf-a0a3-204f170b05c9"/>

?????????????????????????SME(Subject Matter Experts)?????????????Airbnb????????????????????

???*price* ?????????????????????

In [ ]:
fixed_price_df.filter(col("price") == 0).count()

<i18n value="bf195d9b-ea4d-4a3e-8b61-372be8eec327"/>

*price* ?0??????????????????

In [ ]:
pos_prices_df = fixed_price_df.filter(col("price") > 0)

<i18n value="dc8600db-ebd1-4110-bfb1-ce555bc95245"/>

*minimum\_nights* ???? *???* ? *???* ?????????

In [ ]:
display(pos_prices_df.select("minimum_nights").describe())

In [ ]:
display(pos_prices_df
        .groupBy("minimum_nights").count()
        .orderBy(col("count").desc(), col("minimum_nights"))
       )

<i18n value="5aa4dfa8-d9a1-42e2-9060-a5dcc3513a0d"/>

minimum\_nights ????1???????????*minimum\_nights* ?365?????????????????????????

In [ ]:
min_nights_df = pos_prices_df.filter(col("minimum_nights") <= 365)

display(min_nights_df)

<i18n value="25a35390-d716-43ad-8f51-7e7690e1c913"/>

### ???????? (Handling Null Values)

???????????????????????????????????????????????????????????(???????????????????????????????????????)?

??????????????: 
* ?????????????????? 
* ??????:
    * ???/???/??/??????????
* ???????????: 
    * ??????????
    * ????????????
* ?????????????? ALS (Alternating Least Squares) ???????????????
  
**????/???????????(Imputation)??????????????????????????????????????????????(MUST)?**

SparkML?Imputer(??)?????????????????????

<i18n value="83e56fca-ce6d-4e3c-8042-0c1c7b9eaa5a"/>

### ??:Double??????? (Impute: Cast to Double)

SparkML? <a href="https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.Imputer.html?highlight=imputer#pyspark.ml.feature.Imputer" target="_blank">Imputer</a> ????????????double???????????????????????????double????????????

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

integer_columns = [x.name for x in min_nights_df.schema.fields if x.dataType == IntegerType()]
doubles_df = min_nights_df

for c in integer_columns:
    doubles_df = doubles_df.withColumn(c, col(c).cast("double"))

columns = "\n - ".join(integer_columns)
print(f"Columns converted from Integer to Double:\n - {columns}")

<i18n value="69b58107-82ad-4cec-8984-028a5df1b69e"/>

??????NULL?????????????????

In [ ]:
from pyspark.sql.functions import when

impute_cols = [
    "bedrooms",
    "bathrooms",
    "beds", 
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

for c in impute_cols:
    doubles_df = doubles_df.withColumn(c + "_na", when(col(c).isNull(), 1.0).otherwise(0.0))

In [ ]:
display(doubles_df.describe())

<i18n value="c88f432d-1252-4acc-8c91-4834c00da789"/>

### ??????? (Transformers and Estimators)

Spark ML?????????????API????????????????1????????(??????)????????????????Spark ML API??????2????????????????: **`???(transformer)`** ? **`???(estimator)`** .

**???(transformer)**: DataFrame???DataFrame???????DataFrame???????????1?????????????????DataFrame??????Transformer????????????????????????????????????Transformer? **`.transform()`** ??????????

**???(estimator)**: DataFrame??????????????Transformer?????????????????????????????????DataFrame??????????????estimator???Estimator? **`.fit()`** ???????????DataFrame??????????(?????????)????

In [ ]:
from pyspark.ml.feature import Imputer

imputer = Imputer(strategy="median", inputCols=impute_cols, outputCols=impute_cols)

imputer_model = imputer.fit(doubles_df)
imputed_df = imputer_model.transform(doubles_df)

<i18n value="4df06e83-27e6-4cc6-b66d-883317b2a7eb"/>

???????????????????DataFrame?Delta???????????????????

In [ ]:
imputed_df.write.format("delta").mode("overwrite").save(f"{DA.paths.working_dir}/imputed_results")

&copy; 2022 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/>
<a href="https://databricks.com/privacy-policy">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use">Terms of Use</a> | <a href="https://help.databricks.com/">Support</a>